In [1]:
# pipenv install pandas plotly matplotlib pingouin nbformat ipykernel scikit-learn optuna ipywidgets gradio

# EDA
import pandas as pd
import pingouin as pg
import plotly.express as px
import plotly.figure_factory as ff
import matplotlib.pyplot as plt

# ML
from sklearn.model_selection import cross_validate, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Hyperparams otimization
import optuna

# Exploratory Data Aanalysis

## Data load

In [10]:
# Load the Dataset
df_segments = pd.read_csv('./datasets/clients_segments.csv')
df_segments.head(10)

,economic_activity,monthly_revenue,number_of_employees,location,age,innovation,customer_segment
0,Commerce,713109.95,12,Rio de Janeiro,6,1,Bronze
1,Commerce,790714.38,9,São Paulo,15,0,Bronze
2,Commerce,1197239.33,17,São Paulo,4,9,Silver
3,Industry,449185.78,15,São Paulo,6,0,Starter
4,Agribusiness,1006373.16,15,São Paulo,15,8,Silver
5,Services,1629562.41,16,Rio de Janeiro,11,4,Silver
6,Services,771179.95,13,Vitória,0,1,Starter
7,Services,707837.61,16,São Paulo,10,6,Silver
8,Commerce,888983.66,17,Belo Horizonte,10,1,Bronze
9,Industry,1098512.64,13,Rio de Janeiro,9,3,Bronze


In [ ]:
df_segments.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   economic_activity    500 non-null    str    
 1   monthly_revenue      500 non-null    float64
 2   number_of_employees  500 non-null    int64  
 3   location             500 non-null    str    
 4   age                  500 non-null    int64  
 5   innovation           500 non-null    int64  
 6   customer_segment     500 non-null    str    
dtypes: float64(1), int64(3), str(3)
memory usage: 27.5 KB


In [16]:
# Possible values for the categoric columns
print(F"Economic Activity - {df_segments['economic_activity'].unique()} \n")

print(F"Location - {df_segments['location'].unique()}\n")

print(F"Customer segment - {df_segments['customer_segment'].unique()}")

Economic Activity - <StringArray>
['Commerce', 'Industry', 'Agribusiness', 'Services']
Length: 4, dtype: str 

Location - <StringArray>
['Rio de Janeiro', 'São Paulo', 'Vitória', 'Belo Horizonte']
Length: 4, dtype: str

Customer segment - <StringArray>
['Bronze', 'Silver', 'Starter', 'Gold']
Length: 4, dtype: str


## Categoric Variable analysis 

In [25]:
# Creating a sorted list of target
target_list = ['Starter', 'Bronze', 'Silver', 'Gold']

In [ ]:
# Distribution on target variable (Customer segment)
target_count = df_segments.value_counts('customer_segment')
target_count

customer_segment
Silver     260
Bronze     202
Starter     22
Gold        16
Name: count, dtype: int64

In [26]:
px.bar(target_count, color=target_count.index, category_orders={'customer_segment': target_list})

In [28]:
# Percentual distribution on target variable
target_percentual = (target_count / len(df_segments)) * 100
px.bar(target_percentual, color=target_percentual.index, category_orders={'customer_segment': target_list})

In [29]:
location_percentual = (df_segments.value_counts('location') / len(df_segments)) * 100
px.bar(location_percentual, color=location_percentual.index)

In [30]:
economic_percentual = (df_segments.value_counts('economic_activity') / len(df_segments)) * 100
px.bar(economic_percentual, color=economic_percentual.index)

In [33]:
# Contingence Table between Location and Target
location_crosstab = pd.crosstab(df_segments['location'], df_segments['customer_segment'], margins=True)[target_list].reset_index()

location_table = ff.create_table(location_crosstab)

location_table.show()

In [34]:
# Contingence Table between Economic Activity and Target
economic_crosstab = pd.crosstab(df_segments['economic_activity'], df_segments['customer_segment'], margins=True)[target_list].reset_index()

economic_table = ff.create_table(economic_crosstab)

economic_table.show()

In [35]:
# Contingence Table between Innovation and Target
innovation_crosstab = pd.crosstab(df_segments['innovation'], df_segments['customer_segment'], margins=True)[target_list].reset_index()

innovation_table = ff.create_table(innovation_crosstab)

innovation_table.show()

### Location X Customer Segment

In [ ]:
# Pearson's chi-squared test
# H0 - The variables are independent
# H1 - The variables aren't independent
# If pvalue > 0.05 accept H0

expected_value, observed_value, stats = pg.chi2_independence(df_segments, 'customer_segment', 'location')

e:\Programação\Rocketseat\ml_models_practice\04. Decision Tree Classifier\Company Segment Classification\.venv\Lib\site-packages\pingouin\contingency.py:152: UserWarning: Low count on observed frequencies.
  warnings.warn(f"Low count on {name} frequencies.")
e:\Programação\Rocketseat\ml_models_practice\04. Decision Tree Classifier\Company Segment Classification\.venv\Lib\site-packages\pingouin\contingency.py:152: UserWarning: Low count on expected frequencies.
  warnings.warn(f"Low count on {name} frequencies.")


In [45]:
# Expected Value - Frequency expected if there is no relation between the variables, calculated on basis with the distribution used no the chi-squared test
expected_value

location,Belo Horizonte,Rio de Janeiro,São Paulo,Vitória
customer_segment,,,,
Bronze,44.844,52.924,48.884,55.348
Gold,3.552,4.192,3.872,4.384
Silver,57.720,68.120,62.920,71.240
Starter,4.884,5.764,5.324,6.028


In [46]:
# Observed_value - The real frequency on colected data
observed_value

location,Belo Horizonte,Rio de Janeiro,São Paulo,Vitória
customer_segment,,,,
Bronze,39,62,45,56
Gold,4,3,5,4
Silver,63,60,65,72
Starter,5,6,6,5


In [47]:
stats.round(5)

,test,lambda,chi2,dof,pval,cramer,power
0,pearson,1.00000,5.19335,9.0,0.81714,0.05884,0.11369
1,cressie-read,0.66667,5.19198,9.0,0.81726,0.05883,0.11367
2,log-likelihood,0.00000,5.19713,9.0,0.81680,0.05886,0.11374
3,freeman-tukey,-0.50000,5.20798,9.0,0.81581,0.05892,0.11390
4,mod-log-likelihood,-1.00000,5.22494,9.0,0.81428,0.05902,0.11414
5,neyman,-2.00000,5.27777,9.0,0.80945,0.05932,0.11490


The variables location and customer segment are independent. (P-value = 0.81714)

### Economic Activity X Customer Segment

In [ ]:
# Pearson's chi-squared test
# H0 - The variables are independent
# H1 - The variables aren't independent
# If pvalue > 0.05 accept H0

expected_value, observed_value, stats = pg.chi2_independence(df_segments, 'customer_segment', 'economic_activity')

e:\Programação\Rocketseat\ml_models_practice\04. Decision Tree Classifier\Company Segment Classification\.venv\Lib\site-packages\pingouin\contingency.py:152: UserWarning: Low count on observed frequencies.
  warnings.warn(f"Low count on {name} frequencies.")
e:\Programação\Rocketseat\ml_models_practice\04. Decision Tree Classifier\Company Segment Classification\.venv\Lib\site-packages\pingouin\contingency.py:152: UserWarning: Low count on expected frequencies.
  warnings.warn(f"Low count on {name} frequencies.")
e:\Programação\Rocketseat\ml_models_practice\04. Decision Tree Classifier\Company Segment Classification\.venv\Lib\site-packages\scipy\stats\_stats_py.py:7201: RuntimeWarning: divide by zero encountered in power
  terms = f_obs * ((f_obs / f_exp)**lambda_ - 1)
e:\Programação\Rocketseat\ml_models_practice\04. Decision Tree Classifier\Company Segment Classification\.venv\Lib\site-packages\scipy\stats\_stats_py.py:7201: RuntimeWarning: invalid value encountered in multiply
  terms

In [55]:
# Expected Value - Frequency expected if there is no relation between the variables, calculated on basis with the distribution used no the chi-squared test
expected_value

innovation,0,1,2,3,4,5,6,7,8,9
customer_segment,,,,,,,,,,
Bronze,21.008,23.028,22.624,19.392,17.372,17.372,21.816,21.412,18.988,18.988
Gold,1.664,1.824,1.792,1.536,1.376,1.376,1.728,1.696,1.504,1.504
Silver,27.040,29.640,29.120,24.960,22.360,22.360,28.080,27.560,24.440,24.440
Starter,2.288,2.508,2.464,2.112,1.892,1.892,2.376,2.332,2.068,2.068


In [56]:
# Observed_value - The real frequency on colected data
observed_value

innovation,0,1,2,3,4,5,6,7,8,9
customer_segment,,,,,,,,,,
Bronze,36,44,32,22,12,14,15,12,9,6
Gold,0,0,0,0,0,3,0,5,4,4
Silver,10,5,20,25,30,25,38,36,34,37
Starter,6,8,4,1,1,1,1,0,0,0


In [57]:
stats.round(5)

,test,lambda,chi2,dof,pval,cramer,power
0,pearson,1.00000,164.29399,27.0,0.0,0.33095,0.99850
1,cressie-read,0.66667,165.49946,27.0,0.0,0.33216,0.99861
2,log-likelihood,0.00000,181.48878,27.0,0.0,0.34784,0.99951
3,freeman-tukey,-0.50000,NaN,27.0,NaN,NaN,NaN
4,mod-log-likelihood,-1.00000,inf,27.0,0.0,inf,NaN
5,neyman,-2.00000,NaN,27.0,NaN,NaN,NaN


The variables economic activity and customer segment are independent. (P-value = 0.35292)

### Innovation X Customer Segment

In [63]:
# Pearson's chi-squared test
# H0 - The variables are independent
# H1 - The variables aren't independent
# If pvalue > 0.05 accept H0

expected_value, observed_value, stats = pg.chi2_independence(df_segments, 'customer_segment', 'innovation')

e:\Programação\Rocketseat\ml_models_practice\04. Decision Tree Classifier\Company Segment Classification\.venv\Lib\site-packages\pingouin\contingency.py:152: UserWarning: Low count on observed frequencies.
  warnings.warn(f"Low count on {name} frequencies.")
e:\Programação\Rocketseat\ml_models_practice\04. Decision Tree Classifier\Company Segment Classification\.venv\Lib\site-packages\pingouin\contingency.py:152: UserWarning: Low count on expected frequencies.
  warnings.warn(f"Low count on {name} frequencies.")
e:\Programação\Rocketseat\ml_models_practice\04. Decision Tree Classifier\Company Segment Classification\.venv\Lib\site-packages\scipy\stats\_stats_py.py:7201: RuntimeWarning: divide by zero encountered in power
  terms = f_obs * ((f_obs / f_exp)**lambda_ - 1)
e:\Programação\Rocketseat\ml_models_practice\04. Decision Tree Classifier\Company Segment Classification\.venv\Lib\site-packages\scipy\stats\_stats_py.py:7201: RuntimeWarning: invalid value encountered in multiply
  terms

In [64]:
# Expected Value - Frequency expected if there is no relation between the variables, calculated on basis with the distribution used no the chi-squared test
expected_value

innovation,0,1,2,3,4,5,6,7,8,9
customer_segment,,,,,,,,,,
Bronze,21.008,23.028,22.624,19.392,17.372,17.372,21.816,21.412,18.988,18.988
Gold,1.664,1.824,1.792,1.536,1.376,1.376,1.728,1.696,1.504,1.504
Silver,27.040,29.640,29.120,24.960,22.360,22.360,28.080,27.560,24.440,24.440
Starter,2.288,2.508,2.464,2.112,1.892,1.892,2.376,2.332,2.068,2.068


In [65]:
# Observed_value - The real frequency on colected data
observed_value

innovation,0,1,2,3,4,5,6,7,8,9
customer_segment,,,,,,,,,,
Bronze,36,44,32,22,12,14,15,12,9,6
Gold,0,0,0,0,0,3,0,5,4,4
Silver,10,5,20,25,30,25,38,36,34,37
Starter,6,8,4,1,1,1,1,0,0,0


In [66]:
stats.round(5)

,test,lambda,chi2,dof,pval,cramer,power
0,pearson,1.00000,164.29399,27.0,0.0,0.33095,0.99850
1,cressie-read,0.66667,165.49946,27.0,0.0,0.33216,0.99861
2,log-likelihood,0.00000,181.48878,27.0,0.0,0.34784,0.99951
3,freeman-tukey,-0.50000,NaN,27.0,NaN,NaN,NaN
4,mod-log-likelihood,-1.00000,inf,27.0,0.0,inf,NaN
5,neyman,-2.00000,NaN,27.0,NaN,NaN,NaN


The variables innovation and customer segment aren't independent. (P-value = 0.0)

## Numeric Variable analysis

In [40]:
# Distribution company age
px.histogram(df_segments, x="age")

In [41]:
# Distribution monthly revenue
px.histogram(df_segments, x="monthly_revenue")

In [42]:
# Box plot between age and segment
px.box(df_segments, x='customer_segment', y='age', color='customer_segment', category_orders={'customer_segment': target_list})

In [43]:
# Box plot between monthly revenue and segment
px.box(df_segments, x='customer_segment', y='monthly_revenue', color='customer_segment', category_orders={'customer_segment': target_list})